In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, _, _, match) in pinned.items():
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "061a82116be00c585cab41d5b2a790e2a4f96d44"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/061a82116be00c585cab41d5b2a790e2a4f96d44/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Entropy, Cross-Entropy, and KL Divergence

A loss curve is the first thing we look at when a classifier trains, and we
have learned to read its shape: a falling curve is good news, a flat one is
suspicious. The numbers on its axis are another matter. So far we can explain
one of them. A freshly initialized classifier with ten classes reports a loss
of about $2.30$, and that section used this value to check
that an untrained network is set up correctly. The value is $\ln 10$, the cost
of naming one of ten labels when the model still finds all ten equally likely.
The other end of the curve raises harder questions. Is a final loss of $0.6$
good? Is a plateau a bug or a property of the data? If some of the labels are
wrong, how low can any model go?

The loss is the average of $-\log q(y)$, the model's surprise at the label
$y$ that occurred. That average splits into two parts. The first is the
average surprise of someone who knows the true distribution of the labels,
the *entropy*. The data fix it, and no model can reduce it. The second is the
extra surprise of using the wrong distribution, the *Kullback--Leibler (KL)
divergence*. It is never negative, and it is zero only for the right model. A
loss can therefore be judged only against its floor.

Two facts are already in hand from the previous chapter: maximum likelihood
minimizes a cross-entropy (that section), and the KL
divergence is never negative (that section),
which is Gibbs' inequality. There the inequality was a tool. Here it is the
subject. We start from a deck of cards and a source with four symbols, where
entropy counts yes/no questions. We then compare two distributions, and return
to the classifier to find its floor. Finally we ask what changes when the
targets are softened and when the outcomes are real numbers.

The logarithm base is a choice of unit: base $2$ gives *bits*, base $e$ gives
*nats*. **We work in nats throughout**, as the library `log` does and as
training losses are reported. Bits suit yes/no questions, and a subscript $2$
marks a quantity in bits, as in $H_2$ for an entropy. One bit is $\ln 2
\approx 0.693$ nats.
the table collects the symbols of the chapter.

<!-- tbl-caption:Symbols of the information theory chapter. The binary entropy $H_b(p)$ is a function of a number, not of a random variable. The mutual information $I(X;Y)$ and the last five rows are defined in later sections. -->

| Symbol | Meaning | Unit |
|:--|:--|:--|
| $I(x) = -\log p(x)$ | self-information (surprisal) of the outcome $x$ (by convention the letter is shared with the mutual information $I(X;Y)$) | nats |
| $H(X)$, $H(P)$ | entropy of a discrete random variable or distribution; a subscript $2$, as in $H_2$ and $\textrm{CE}_2$, marks bits | nats |
| $H_b(p)$ | binary entropy of a coin with heads probability $p$ | nats |
| $h(X)$ | differential entropy of a continuous random variable | nats, depends on the units of $X$ |
| $D_{\textrm{KL}}(P \Vert Q)$ | KL divergence from $P$ to $Q$ | nats |
| $\textrm{CE}(P, Q)$ | cross-entropy from $P$ to $Q$ | nats |
| $k$ | number of classes (size of the set of outcomes) | |
| $n$ | number of examples, or of symbols in a message; the dimension in $\mathbb{R}^n$ | |
| $\mathbf{p}$, $\mathbf{q}$ | probability vectors on $k$ classes (truth, model) | |
| $\mathbf{z}$ | logits, with $\mathbf{q} = \mathrm{softmax}(\mathbf{z})$ | |
| $\epsilon$ | label-smoothing mass (in later sections, a noise variable) | |
| $\tau$ | temperature | |
| $\rho$ | label-noise rate: the probability that a recorded label is not the clean class | |
| $H(X \mid Y)$, $I(X;Y)$ | conditional entropy, mutual information | nats |
| $D_f(P \Vert Q)$ | f-divergence with divergence function $f$ | |
| $T$ | critic: a function whose averages under two distributions bound a divergence | |
| $g$ | test function of an integral probability metric | |
| $\gamma$ | coupling (transport plan) of two distributions | |
| $N$ | number of points in each of two samples that are compared; number of candidates in a contrastive batch | |

In [1]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## Information and Entropy

### Self-Information

Consider reports about a shuffled deck of cards. "I see a card" conveys
nothing, because the event is certain. "I see a heart" picks one of four
equally likely suits, "It is the three of spades" one of $52$ equally likely
cards, and the order of the whole deck one of $52!$ orderings. The less
probable an event, the more we learn by observing it. Shannon
[@Shannon.1948] measured this by the *self-information* of an outcome $x$
with probability $p(x)$,

$$
I(x) = -\log p(x).
$$

It is also called the *surprisal* of the outcome, the name used in
that section: the less we expected an outcome, the more
it surprises us. The surprise belongs to whoever assigned the probability, so
a confidently wrong model finds ordinary events surprising. A certain event
carries no information, and halving the probability of an outcome adds $\ln
2$ nats, one bit, to its surprisal. The first three card reports carry $0$,
$\ln 4 \approx 1.39$ and $\ln 52 \approx 3.95$ nats. In bits, where one bit
is the information of one fair coin flip, they carry $0$, $2$ and $5.70$.

Why a logarithm? We want three properties: a certain event carries zero
information, rarer events carry more, and the information of two independent
events adds up. Since independent probabilities multiply, the third property
says $I(p_1 p_2) = I(p_1) + I(p_2)$, and the only decreasing functions with
this property are multiples of $-\log p$ (@Csiszar.2008;
Exercise 1 gives
the argument). The choice of multiple is the choice of unit.

![Self-information $I(x) = -\log p(x)$ in nats as a function of the probability $p$. Certain events ($p = 1$) carry zero information, the fair coin ($p = \tfrac{1}{2}$) carries $\ln 2 \approx 0.693$ nats, and the curve diverges as $p \to 0$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-it-self-info-curve.svg)

the figure plots the curve: decreasing, convex, zero at
$p = 1$, and diverging as $p \to 0$. The fourth card report, the order of the
deck, lies far out on it, at $\ln 52!$. We compute this value through the
log-gamma function, $\ln 52! = \ln \Gamma(53)$, without ever forming the
68-digit number $52!$.

In [2]:
deck_order = math.lgamma(53)          # ln 52! without forming 52!
print(f'the order of the deck: {deck_order:.2f} nats '
      f'= {deck_order / math.log(2):.1f} bits')

the order of the deck: 156.36 nats = 225.6 bits


Naming the order of a shuffled deck carries $156$ nats, as much information as
$226$ fair coin flips.

### Entropy

A surprisal is known only once the outcome has arrived. Before that we can
say how surprised we expect to be, by averaging the surprisal over the
outcomes. Let $X$ be a discrete random variable that takes the value $x_i$
with probability $p_i$. Its *entropy* is the expected self-information of its
outcome,

$$
H(X) = -\mathbb{E}_{x \sim P}[\log p(x)] = -\sum_i p_i \log p_i,
$$

with the convention $0 \log 0 = 0$, justified by continuity ($t \log t \to 0$
as $t \to 0$): an impossible outcome contributes nothing. Since the entropy
depends only on the distribution $P$ of $X$, we write $H(P)$ as well as $H(X)$.

On a finite set, entropy in bits has a concrete meaning. It counts the yes/no
questions that a good questioner needs, on average, to identify the outcome.
Take the distribution $P = (\tfrac12, \tfrac14, \tfrac18, \tfrac18)$ on four
symbols $a, b, c, d$. We carry it through the section and call it the *dyadic
source*, since all its probabilities are powers of $\tfrac12$. A good
questioner first asks "Is it $a$?", which settles the
matter half the time. If not, they ask "Is it $b$?", and then "Is it $c$?",
where "no" means $d$. The expected number of questions is

$$
\tfrac12 \cdot 1 + \tfrac14 \cdot 2 + \tfrac18 \cdot 3 + \tfrac18 \cdot 3
= 1.75,
$$

and the entropy in bits is the same number,
$H_2(P) = \tfrac12 \log_2 2 + \tfrac14 \log_2 4 + 2 \cdot \tfrac18 \log_2 8
= 1.75$ bits, or $1.213$ nats. Each outcome is identified with
$\log_2(1/p_i)$ questions, its self-information in bits.

A questioner who ignores the probabilities asks "Is it $a$ or $b$?" and then
"Which of the two?", and always needs $2$ questions. A questioning strategy is
a binary code in disguise: write $1$ for yes and $0$ for no, and the answers
that identify an outcome form its codeword, here $1$, $01$, $001$ and $000$ for
$a$, $b$, $c$ and $d$. The count of questions matched the entropy here because
every probability is a power of $\tfrac12$. In the language of codes,
that section proves that the count is never far off:
for every distribution on a finite set, the best strategy needs between $H_2$
and $H_2 + 1$ questions on average.

In code, the function `entropy` masks the zero probabilities, which implements
$0 \log 0 = 0$. We use it for the rest of the section, first on the dyadic
source.

In [3]:
def entropy(p):
    """Entropy of a probability vector, in nats (0 log 0 = 0 by masking)."""
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-np.dot(p, np.log(p)))

p_dyadic = np.array([1 / 2, 1 / 4, 1 / 8, 1 / 8])
h = entropy(p_dyadic)
print(f'H = {h:.3f} nats = {h / math.log(2):.2f} bits')

H = 1.213 nats = 1.75 bits


It returns the $1.75$ bits that we counted in questions.

Every term $-p_i \log p_i$ is non-negative because $0 \le p_i \le 1$, so $H(X)
\ge 0$, with equality only for a point mass, a variable whose value is certain.
The entropy also depends only on the list of probabilities, not on the values
or names of the outcomes. A variable that takes the values $1$, $100$, $10^6$,
and $10^9$ with the dyadic probabilities also has $1.75$ bits.

### Joint and Conditional Entropy

A label comes with an input, and a character with the text before it. We
therefore need the entropy of two variables together, and the entropy left in
one of them once the other is known. The first needs nothing new. A pair
$(X, Y)$ with joint mass function $p(x, y)$ is a single random variable
whose outcomes are pairs, and the entropy of the pair is the *joint entropy*
$H(X, Y) = -\mathbb{E}[\log p(X, Y)]$. The *conditional entropy* is the
surprisal left in $Y$ once $X$ is known, averaged over both,

$$
H(Y \mid X) = -\mathbb{E}[\log p(Y \mid X)] = \sum_x p(x)\, H(Y \mid X = x),
$$

where $H(Y \mid X = x)$ is the entropy of the conditional distribution $p(\cdot
\mid x)$. Since $\log p(x, y) = \log p(x) + \log p(y \mid x)$, taking
expectations gives the *chain rule*

$$
H(X, Y) = H(X) + H(Y \mid X).
$$

The entropy of a pair is the entropy of the first variable plus what the
second adds once the first is known. Applied repeatedly, the rule reads $H(X_1,
\ldots, X_n) = \sum_{i=1}^n H(X_i \mid X_1, \ldots, X_{i-1})$: the entropy of a
sequence is the sum of the entropies of its symbols, each given the symbols
before it. A language model predicts text in this order, one symbol at a time.

The questions game is an instance. Let $X$ be the answer to the first question,
"Is it $a$?", and let $Y$ be the outcome of the dyadic source. The answer is a
fair coin, so $H_2(X) = 1$ bit. After "yes" nothing is left to ask. After "no"
the outcome is $b$, $c$ or $d$ with probabilities $(\tfrac12, \tfrac14,
\tfrac14)$, a distribution with $1.5$ bits, so $H_2(Y \mid X) = \tfrac12 \cdot
0 + \tfrac12 \cdot 1.5 = 0.75$ bits. The chain rule gives $H_2(X, Y) = 1 +
0.75 = 1.75$ bits: the first answer supplies one bit and the later questions
the rest. The sum is the entropy of the outcome alone, as it has to be. The
outcome determines the answer, so $H(X \mid Y) = 0$, and the chain rule in the
other order reads $H(X, Y) = H(Y) + 0$.

### Maximum Entropy on a Finite Set

How large can the entropy of a variable with $k$ outcomes be? We start with the
smallest case, a coin. A Bernoulli variable with heads probability $p$ has the
*binary entropy*

$$
H_b(p) = -p \log p - (1-p) \log(1-p),
$$

a function of the number $p$ rather than of a random variable.
the figure plots it. The entropy is zero at
$p \in \{0, 1\}$, where the outcome is certain, and largest at the fair coin,
$H_b(\tfrac12) = \ln 2 \approx 0.693$ nats, one bit. A coin with $p = 0.1$
has $H_b(0.1) \approx 0.325$ nats, because its outcome is easy to guess.

![The binary entropy $H_b(p)$ in nats, symmetric about $p = \tfrac{1}{2}$, where it peaks at $\ln 2 \approx 0.693$ nats. At the deterministic endpoints $p = 0$ and $p = 1$ the outcome carries no surprise and the entropy vanishes. The curve is concave: any chord lies below it.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-it-bernoulli-entropy.svg)

The curve is concave: the chord joining any two of its points lies below it. It
peaks at the fair coin, and the same holds for any number of outcomes. If $X$
takes at most $k$ values, then

$$
H(X) \leq \log k,
$$

with equality if and only if $X$ is uniform on $k$ values. The uniform
distribution hedges the most, so it is the hardest to guess. For four outcomes
the bound is $\log_2 4 = 2$ bits, $0.25$ bits above the $1.75$ bits of the
dyadic source. We postpone the proof until Gibbs' inequality is available,
where it takes one line. That inequality compares two distributions, and so we
turn to the comparison first.

## KL Divergence and Cross-Entropy

### The Kullback--Leibler Divergence

We now want to measure how badly a model $Q$ misrepresents a truth $P$.
Information theory answers with the *Kullback--Leibler (KL) divergence*, also
called *relative entropy* [@Kullback.Leibler.1951]: the average extra
surprisal of someone who believes $Q$ when the outcomes are drawn from $P$. In
the questions game, the KL divergence in bits is the average number of extra
questions that a strategy designed for $Q$ needs on outcomes drawn from $P$,
up to rounding to whole questions. Let $P$ and $Q$ have mass functions (or
densities) $p$ and $q$. The KL divergence from $P$ to $Q$ is

$$
D_{\textrm{KL}}(P\|Q) = \mathbb{E}_{x \sim P} \left[ \log \frac{p(x)}{q(x)} \right].
$$

The term inside the expectation is
$\log p(x) - \log q(x) = (-\log q(x)) - (-\log p(x))$: the surprisal of the
believer in $Q$ minus the surprisal of the believer in $P$. It is positive
where $Q$ underestimates the outcome ($q(x) < p(x)$) and negative where $Q$
overestimates it. KL averages this difference over outcomes drawn from the
truth.

The definition is asymmetric. $P$ supplies both the samples and the numerator,
so in general $D_{\textrm{KL}}(P\|Q) \neq D_{\textrm{KL}}(Q\|P)$. Outcomes
with $p(x) = 0$ contribute nothing, by the convention $0 \log(0/q) = 0$. If
some outcome has $p(x) > 0$ but $q(x) = 0$, however, then
$D_{\textrm{KL}}(P\|Q) = +\infty$.

A small pair makes the definition concrete, and we carry it through the
section. Let $P = (0.6, 0.3, 0.1)$ be the truth and $Q = (0.2, 0.5, 0.3)$ a
model on three outcomes. Then

$$
D_{\textrm{KL}}(P\|Q)
= 0.6 \ln\frac{0.6}{0.2} + 0.3 \ln\frac{0.3}{0.5} + 0.1 \ln\frac{0.1}{0.3}
= 0.659 - 0.153 - 0.110 = 0.396 \textrm{ nats},
$$

and the reverse direction gives $D_{\textrm{KL}}(Q\|P) = 0.365$ nats. In bits
the two directions are $0.571$ and $0.527$. They differ, and the first says
that a questioner who believes $Q$ asks $0.57$ more questions per outcome than
necessary, on average over long runs of outcomes drawn from $P$.

Here is the discrete case in code, with $0 \log 0 = 0$ implemented by masking
the zero probabilities of $P$, as in `entropy`. We call it on the running pair
in both directions, and on a model $Q' = (0.5, 0.5, 0)$ that declares the third
outcome impossible.

In [4]:
def kl_divergence(p, q):
    """KL(P || Q) for two probability vectors, in nats; inf where q = 0 < p."""
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    m = p > 0
    if np.any(q[m] == 0):
        return float('inf')
    return float(np.dot(p[m], np.log(p[m] / q[m])))

p = np.array([0.6, 0.3, 0.1])
q = np.array([0.2, 0.5, 0.3])
print(f'KL(P||Q) = {kl_divergence(p, q):.4f} nats, '
      f'KL(Q||P) = {kl_divergence(q, p):.4f} nats')
print(f"KL(P||Q') = {kl_divergence(p, [0.5, 0.5, 0.0])}")

KL(P||Q) = 0.3961 nats, KL(Q||P) = 0.3653 nats
KL(P||Q') = inf


The second line is the one to remember. The model $Q'$ gives zero probability
to an outcome that occurs one time in ten, and its divergence is infinite: a
model that declares an event impossible is infinitely surprised when it
happens.

### Gibbs' Inequality

Can the extra surprisal be negative? That would mean a wrong model is less
surprised, on average, than the truth itself. Gibbs' inequality
[@Gibbs.1902] says that it cannot: no model beats the truth on average.
Everything that follows rests on it, so we prove it in full, as a statement
about averaging a concave function. For a concave function $f$ and a random
variable $X$ with finite mean, *Jensen's inequality* states that

$$
\mathbb{E}[f(X)] \le f(\mathbb{E}[X]).
$$

Averaging first and applying $f$ second gives the larger value. For an $X$ with
two values, this is the chord below the curve of
the figure. For strictly concave $f$, equality holds
only when $X$ is constant with probability one. that section
proves the general case. We need it for one function, the logarithm, which is
strictly concave: the logarithm of an average is at least the average of the
logarithms.

**Proposition (Gibbs' inequality).** *For any distributions $P$ and $Q$ on
the same space,*

$$
D_{\textrm{KL}}(P\|Q) \geq 0,
$$

*with equality if and only if $P = Q$.*

**Proof.** If $q(x) = 0$ for some $x$ with $p(x) > 0$, the divergence is
$+\infty$ and there is nothing to prove, so assume otherwise. Write the
divergence as minus the average of $\log(q(x)/p(x))$ over $x \sim P$, and
apply Jensen's inequality to that logarithm:

$$
\begin{aligned}
D_{\textrm{KL}}(P\|Q)
&= -\mathbb{E}_{x\sim P}\!\left[\log\frac{q(x)}{p(x)}\right]
\geq -\log \mathbb{E}_{x\sim P}\!\left[\frac{q(x)}{p(x)}\right] \\
&= -\log \!\!\sum_{x\,:\,p(x)>0}\!\! q(x)
= -\log Q(\textrm{supp}\,P)
\geq 0.
\end{aligned}
$$

The minus sign turns Jensen's "at most" into "at least". Everything else
happens in the second line. The expectation runs over the support of $P$, the
set of outcomes with $p(x) > 0$. The factors $p(x)$ cancel, and what remains
is the $Q$-mass of that support, a number at most $1$, whose negative logarithm
is non-negative. For densities, the same computation runs with integrals. For
equality, both inequalities must be tight. Because the logarithm is strictly
concave, the first is an equality only when $q(x)/p(x)$ equals a constant $c$
for $P$-almost every $x$ (for all $x$ outside a set of $P$-probability zero),
and the second requires $Q(\textrm{supp}\,P) = 1$. Summing (or integrating) $q
= c\,p$ over the support of $P$ then gives $c = 1$, so $q = p$, in the case of
densities again outside a set of probability zero. $\blacksquare$

The proof has one idea. Averaged over the truth, the ratio $q/p$ is at most
one, so its logarithm averages to at most zero, with equality only when there
is nothing to average. A model incurs no extra surprisal only when it is
right. Exercise 2 compares this
proof with the tangent-line proof.

The inequality settles the uniform bound that we left unproved. For a
distribution $P$ on $k$ outcomes and the uniform distribution $U$ on the same
outcomes, $D_{\textrm{KL}}(P\|U) = \sum_i p_i \log(k\,p_i) = \log k - H(P)$,
which is non-negative and zero only for $P = U$.

It also shows that conditioning cannot increase entropy. Compare the joint
distribution of a pair with the product of its marginals. The log-ratio is
$\log \frac{p(x, y)}{p(x)\,p(y)} = \log p(y \mid x) - \log p(y)$, and its
average over the joint distribution is $H(Y) - H(Y \mid X)$. That average is a
KL divergence, so

$$
H(Y) - H(Y \mid X) = D_{\textrm{KL}}(p_{X,Y} \,\|\, p_X\, p_Y) \ge 0,
$$

with equality only when $X$ and $Y$ are independent. On average, knowing $X$
never makes $Y$ more uncertain. A particular observation can. Let $Y$ be tails
for certain when $X = 0$, and a fair coin when $X = 1$, which happens one time
in five. Then $Y$ shows heads with probability $0.1$, and $H(Y) = H_b(0.1)
\approx 0.325$ nats. Observing $X = 1$ raises the uncertainty about $Y$ to $\ln
2 \approx 0.693$ nats, yet the average $H(Y \mid X) = 0.2 \cdot 0.693 \approx
0.139$ nats is less than half of $H(Y)$.

### Cross-Entropy

Classifiers are trained with a close relative of the KL divergence. The
*cross-entropy* from $P$ to $Q$ is the average surprisal of the
believer in $Q$ on outcomes drawn from $P$,

$$
\textrm{CE}(P, Q) = -\mathbb{E}_{x \sim P} [\log q(x)].
$$

Adding and subtracting $\log p(x)$ inside the expectation, which is
legitimate whenever $H(P)$ is finite, splits the model's surprisal into the
truth's own surprisal plus the excess:

$$
\textrm{CE}(P, Q) = H(P) + D_{\textrm{KL}}(P\|Q).
$$

The cross-entropy is an entropy floor plus a never-negative excess: by Gibbs'
inequality, $\textrm{CE}(P,Q) \ge H(P)$ with equality if and only if $Q = P$.
For the running pair, $\textrm{CE}(P, Q) = -(0.6 \ln 0.2 + 0.3 \ln 0.5 + 0.1
\ln 0.3) = 1.294$ nats and $H(P) = 0.898$ nats; their difference, $0.396$
nats, is the KL divergence computed above. Since $H(P)$ does not depend on
$Q$, minimizing the cross-entropy in $Q$ is minimizing the KL divergence.
Maximum likelihood on discrete data does just that: its average negative
log-likelihood is the cross-entropy from the empirical distribution of the
data to the model (that section).

the figure draws the decomposition in the questions
game, for the dyadic source and the uniform model $U = (\tfrac14, \tfrac14,
\tfrac14, \tfrac14)$. Asking adaptively takes $H_2(P) = 1.75$ questions on
average. The two fixed questions of the uniform questioner take
$\textrm{CE}_2(P, U) = 2$, and the difference, $D_{\textrm{KL}}(P\|U) = 0.25$
bits, is the waste. Since a questioning strategy is a code, the same numbers
are average codeword lengths in bits per symbol. For this pair the reverse
divergence $D_{\textrm{KL}}(U\|P)$ also happens to be $0.25$ bits, which is why
we showed the asymmetry of KL on the other pair.

![The decomposition the equation for the dyadic source $P = (\tfrac{1}{2}, \tfrac{1}{4}, \tfrac{1}{8}, \tfrac{1}{8})$ and the uniform model $U$, in bits. Asking the questions adapted to $P$ takes $H_2(P) = 1.75$ questions on average, asking two fixed questions takes $\textrm{CE}_2(P, U) = 2.0$, and the waste is $D_{\textrm{KL}}(P\|U) = 0.25$ bits.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-it-code-length-bars.svg)

## The Classification Loss

In a $k$-class classification problem, the target for one example is the
one-hot distribution that puts all its mass on the recorded label $y$, and the
model supplies a predicted distribution $\mathbf{q} = (q_1, \ldots, q_k)$.
The cross-entropy the equation between them collapses to a single
term, $-\log q_y$, the model's surprisal at the label. Averaged over
$n$ examples with labels $y_i$ and predictions $\mathbf{q}_i$, it is the
*cross-entropy loss*

$$
\textrm{CE} = -\frac{1}{n} \sum_{i=1}^n \log q_{i, y_i},
$$

in nats. A built-in loss computes this number from the *logits* $\mathbf{z}$,
the unnormalized scores whose softmax is $\mathbf{q}$. It applies a stable
log-softmax itself, because forming the probabilities first overflows for
large logits and ends in the logarithm of zero for very negative ones
(that section;
that section derives the stable form). The subsections
that follow read that number.

### The Irreducible Loss

The opening asked how low the loss can go when some labels are wrong. On a
fixed training set the answer is disappointing. A network large enough to
memorize the set can fit every recorded label, the wrong ones included, and
push the training loss toward zero. The question has content only for examples
that the model has not seen, and there the one-hot target is not the truth. In
real data the label is not a function of the input: the same image can carry
different labels, from ambiguity or from annotation errors. The truth about an
input $\mathbf{x}$ is the distribution from which its label is drawn, and the
recorded label is one draw from it.

Let the input $\mathbf{x}$ and the label $y \in \{1, \ldots, k\}$ have a
joint distribution, let $p(\cdot \mid \mathbf{x})$ be the true conditional
distribution of the label, and let $q(\cdot \mid \mathbf{x})$ be any model's
prediction. As the number of fresh examples grows, their average loss
approaches its expectation over the pair $(\mathbf{x}, y)$. We call this
expectation the *population loss*. No model can bring it below the average
entropy of the label given the input.

**Proposition (the irreducible classification loss).** *The population
cross-entropy loss of any model satisfies*

$$
\mathbb{E}_{(\mathbf{x}, y)}\big[-\log q(y \mid \mathbf{x})\big]
= \mathbb{E}_{\mathbf{x}}\big[H(p(\cdot \mid \mathbf{x}))\big]
+ \mathbb{E}_{\mathbf{x}}\big[D_{\textrm{KL}}\big(p(\cdot \mid \mathbf{x}) \,\|\, q(\cdot \mid \mathbf{x})\big)\big]
\ge \mathbb{E}_{\mathbf{x}}\big[H(p(\cdot \mid \mathbf{x}))\big],
$$

*with equality if and only if $q(\cdot \mid \mathbf{x}) = p(\cdot \mid
\mathbf{x})$ for almost every $\mathbf{x}$, that is, for all inputs outside a
set of probability zero.*

**Proof.** Condition on $\mathbf{x}$. The inner expectation over $y$ is the
cross-entropy $\textrm{CE}(p(\cdot \mid \mathbf{x}), q(\cdot \mid
\mathbf{x}))$, and the decomposition the equation applies at each
$\mathbf{x}$; the entropies are finite, at most $\log k$. Averaging over
$\mathbf{x}$ gives the identity. Gibbs' inequality makes every KL term
non-negative, and an average of non-negative terms is zero only when almost
all of them vanish. $\blacksquare$

The floor is the conditional entropy of the label given the input
(that section). It is a property of the data:
no architecture, data volume, or training time lowers it. Of the results of
this section, it is the one to have in mind when reading a loss curve. Three
cases show what it means.

* A model that predicts the uniform distribution has cross-entropy $\ln k$ at
  every input, whatever the labels are: the $2.30$ of the opening for $k = 10$.
* If the label is a function of the input, every $H(p(\cdot \mid \mathbf{x}))$
  is zero, and a perfect model reports $0$.
* Under symmetric label noise, the clean class is determined by the input,
  and the observed label is the clean class with probability $1 - \rho$ and
  each of the other $k - 1$ classes with probability $\rho/(k-1)$. Then
  $H = -(1-\rho)\log(1-\rho) - \rho\log\tfrac{\rho}{k-1} = H_b(\rho) + \rho\log(k-1)$
  at every input, which for $\rho = 0.1$ and $k = 10$ is
  $0.325 + 0.220 = 0.545$ nats. As long as $1 - \rho > \rho/(k-1)$, the best
  prediction of the class is the clean one, so the best accuracy is
  $1 - \rho$, here $90\%$. Exercise
  4 treats inputs
  whose clean class is itself uncertain.

Before running the next cell, guess the loss of a predictor that puts $0.99$ on
the clean class in the noisy case. It is right nine times in ten, and nearly
certain each time. The cell compares it with the uniform predictor and with the
predictor that reports the true distribution, $q = p$, which is called the
*Bayes predictor*.

In [5]:
k, rho = 10, 0.1
p_noisy = np.array([1 - rho] + [rho / (k - 1)] * (k - 1))  # clean class first
predictors = {'uniform': np.full(k, 1 / k),
              'confident 0.99': np.array([0.99] + [0.01 / (k - 1)] * (k - 1)),
              'Bayes q = p': p_noisy}
print(f'floor = H(p) = {entropy(p_noisy):.3f} nats')
for name, q_hat in predictors.items():
    ce = float(-np.dot(p_noisy, np.log(q_hat)))
    print(f'{name:15s} CE = {ce:.3f} nats')

floor = H(p) = 0.545 nats
uniform         CE = 2.303 nats
confident 0.99  CE = 0.689 nats
Bayes q = p     CE = 0.545 nats


The confident predictor reports $0.689$, well above the floor: every flipped
label surprises it, and its confidence loses more on the tenth of the labels
that are flipped than it gains on the rest. Only the Bayes predictor reaches
the floor.

The opening asked whether a final loss of $0.6$ is good. On clean labels,
where the floor is zero, it leaves room for a better model. On these noisy
labels it lies within $0.06$ nats of the floor and leaves almost none. A
plateau at the floor is a property of the data, and only a plateau above it
is the fault of the model or of its training. All of this holds for fresh
data. There a loss below the floor cannot come from a better model, and
Exercise 3 asks what it
signals instead.

The floor was derived for one penalty, $-\log q_y$, for labels as the data
deliver them, and for outcomes that can be counted. The rest of the section
changes these one at a time: other penalties, softened targets, and outcomes
that are real numbers.

### Proper Scoring Rules

The floor is reached only by the true conditional probabilities, which is
what we want of a loss whose outputs we read as probabilities. The property
has a name. A *scoring rule* assigns a penalty $S(\mathbf{q}, y)$ for having
reported the probability vector $\mathbf{q}$ when class $y$ then occurs. The
rule is
*proper* if truthful reporting is optimal: when the labels follow $\mathbf{p}$,
the expected penalty $\mathbb{E}_{y \sim \mathbf{p}}[S(\mathbf{q}, y)]$ is
minimized by reporting $\mathbf{q} = \mathbf{p}$. It is *strictly proper* if
this report is the unique optimum. The cross-entropy loss is the *log score*
$S(\mathbf{q}, y) = -\log q_y$, and it is strictly proper, since its expected
penalty is the cross-entropy $\textrm{CE}(\mathbf{p}, \mathbf{q})$, which is
smallest only at $\mathbf{q} = \mathbf{p}$.

At the population optimum over a model class that contains the truth, the
predicted probabilities are therefore the true conditional probabilities. In
particular they are *calibrated*: among the inputs that receive probability
$0.8$ for a class, that class occurs $80\%$ of the time. A finite sample or a
restricted model class can still leave a trained model miscalibrated.

The log score is not the only strictly proper rule. The *Brier score*
$S(\mathbf{q}, y) = \sum_j (q_j - \mathbf{1}(j = y))^2$ is strictly proper
too, with a bounded penalty on confident errors where the log score's
penalty diverges [@Gneiting.Raftery.2007]. Exercise
6 computes its
excess penalty and tests an improper rule.

### Label Smoothing

So far the target was the one-hot label. A softer target changes what the
loss demands of the logits and where its floor lies, and there is a reason to
want one: one-hot targets ask for something that a softmax cannot deliver.
Probability $1$ on the true class needs a logit infinitely far above the
others, so the loss has no minimizer, and training pushes the logits apart for
as long as it runs. *Label smoothing*
[@Szegedy.Vanhoucke.Ioffe.ea.2016] replaces the one-hot target
$\mathbf{e}_y$ by a mixture with a fraction $\epsilon$ of the uniform
distribution over all $k$ classes,

$$
\mathbf{p}^\epsilon = (1 - \epsilon)\, \mathbf{e}_y + \frac{\epsilon}{k}\, \mathbf{1}.
$$

The true class gets target probability $1 - \epsilon + \epsilon/k$ and each
other class $\epsilon/k$. The training loss is the ordinary cross-entropy
$\textrm{CE}(\mathbf{p}^\epsilon, \mathbf{q})$ to this target, the smoothed
loss of that section. It is smallest at $\mathbf{q} =
\mathbf{p}^\epsilon$, and a softmax reaches that prediction with a finite gap
between the logit of the true class and each other logit,

$$
z_y - z_j = \log\frac{1 - \epsilon + \epsilon/k}{\epsilon/k}
= \log\!\Big(1 + \frac{k(1-\epsilon)}{\epsilon}\Big).
$$

For $k = 10$ classes and $\epsilon = 0.1$ the gap is $\ln 91 \approx 4.51$,
and the loss at the optimum is the entropy of the target,
$H(\mathbf{p}^\epsilon) = 0.500$ nats.

It is instructive to see how fast the logits run away without smoothing. Take
one example, start its logits at zero, and descend the loss with step size
$1$. The gradient with respect to the logits is $\mathbf{q} -
\mathbf{p}^\epsilon$. With a one-hot target the wrong classes stay tied, and a
step widens the gap $g$ between the true class and the rest by $(1 - q_y) +
q_j = k/(e^g + k - 1)$, about $k e^{-g}$ once the gap is large. A step of that
size multiplies $e^g$ by $e^{k e^{-g}} \approx 1 + k e^{-g}$, which adds
about $k$ to it. So $e^g \approx kt$ and $g \approx \ln(kt)$ after $t$
steps. The gap never stops growing, but ten times the training adds only $\ln
10 \approx 2.30$ to it. After $10{,}000$ steps we expect $\ln 10^5 \approx
11.51$. Here is the descent, with and without smoothing.

In [6]:
k, checkpoints = 10, (10, 100, 1000, 10000)
for eps in (0.0, 0.1):
    p_eps = np.full(k, eps / k)
    p_eps[0] += 1 - eps                   # the true class first
    z = np.zeros(k)                       # the logits of one example
    for step in range(1, checkpoints[-1] + 1):
        q = np.exp(z - z.max())
        z -= q / q.sum() - p_eps          # gradient step, step size 1
        if step in checkpoints:
            log_q = z - z.max() - np.log(np.exp(z - z.max()).sum())
            print(f'eps = {eps}  step {step:5d}  gap = {z[0] - z[1]:5.2f}  '
                  f'loss = {-p_eps @ log_q:.4f}')
print(f'optimum for eps = 0.1: gap = ln 91 = {math.log(91):.2f}, '
      f'loss = H(p_eps) = {entropy(p_eps):.4f}')

eps = 0.0  step    10  gap =  4.29  loss = 0.1158
eps = 0.0  step   100  gap =  6.87  loss = 0.0093
eps = 0.0  step  1000  gap =  9.21  loss = 0.0009
eps = 0.0  step 10000  gap = 11.51  loss = 0.0001
eps = 0.1  step    10  gap =  3.81  loss = 0.5243
eps = 0.1  step   100  gap =  4.51  loss = 0.5003
eps = 0.1  step  1000  gap =  4.51  loss = 0.5003


eps = 0.1  step 10000  gap =  4.51  loss = 0.5003
optimum for eps = 0.1: gap = ln 91 = 4.51, loss = H(p_eps) = 0.5003


With one-hot targets the gap reads $4.29$, $6.87$, $9.21$ and $11.51$. The
prediction $\ln(kt)$ is $4.61$, $6.91$, $9.21$ and $11.51$ at these steps: the
gap starts a little below it, where $k e^{-g}$ is still a rough approximation,
and agrees with it to two decimals from $1{,}000$ steps on. The loss keeps
falling without ever arriving at zero. In a network large enough to fit its
training set, the same growth shows as weights that keep growing and as
probabilities near $1$ on every training label, the wrong ones included. With
$\epsilon = 0.1$ the gap is $4.51$ after $100$ steps and stays there, and the
loss stops at $0.500$.

Training can now come to rest, but a smoothed model has to be read
differently. Its training loss levels off at $H(\mathbf{p}^\epsilon)$, not at
zero, even when it predicts the target without error. Its probabilities are
no longer the true conditional ones: no class receives more than $1 - \epsilon
+ \epsilon/k$, here $0.91$. In practice $\epsilon = 0.1$ is the usual choice,
and the recipe of that section uses it. Exercise
7 asks what smoothing does to
the logits of the wrong classes.

### Knowledge Distillation

A soft target can also come from another model. *Knowledge distillation*
[@Hinton.Vinyals.Dean.2015] trains a small *student* network to imitate
a large *teacher*, and that section applies it to language
models. The target is the teacher's whole distribution over the classes, not
only its most probable class. That a "2" looks somewhat like a "7" is
information a one-hot label does not carry. It is stored in the teacher's
logits for the wrong classes, and Exercise
7 asks why a teacher trained
with label smoothing passes on less of it.

As with any target, the loss is smallest when the student reports the target
itself, so the decomposition the equation has nothing new to say
here. What needs an explanation is the scale of the loss: as we are about to
see, it carries a factor that looks arbitrary at first sight.

A trained teacher's softmax is nearly one-hot, and the probabilities it gives
to the wrong classes are too small to influence the student. Both
distributions are therefore first *softened*: the logits are divided by a
*temperature* $\tau > 0$ before the softmax,

$$
p_j^{(\tau)} = \mathrm{softmax}(\mathbf{z}^{\textrm{tea}}/\tau)_j, \qquad
q_j^{(\tau)} = \mathrm{softmax}(\mathbf{z}^{\textrm{stu}}/\tau)_j.
$$

A temperature above one shrinks the gaps between logits and moves
probability toward the wrong classes. The distillation loss is the KL
divergence between the softened distributions, scaled by $\tau^2$:

$$
\mathcal{L}_{\textrm{distill}}
= \tau^2\, D_{\textrm{KL}}\big(\mathbf{p}^{(\tau)} \,\|\, \mathbf{q}^{(\tau)}\big).
$$

The loss is zero only when the student matches the softened teacher. To see
where the factor $\tau^2$ comes from, differentiate the KL term with respect
to a student logit. Only $-\sum_j p_j^{(\tau)} \log q_j^{(\tau)}$ depends on
the student, and the derivative of a log-softmax at temperature $\tau$ is the
usual softmax-minus-target expression scaled by $1/\tau$:

$$
\frac{\partial}{\partial z^{\textrm{stu}}_j}
D_{\textrm{KL}}\big(\mathbf{p}^{(\tau)} \| \mathbf{q}^{(\tau)}\big)
= \frac{1}{\tau}\big(q_j^{(\tau)} - p_j^{(\tau)}\big).
$$

Each student logit is pushed by the gap between the two softened
probabilities of its class, divided by $\tau$.

The gap itself shrinks as well. For large $\tau$ the softened distributions
flatten toward uniform. Subtract the mean logit $\bar z$, which leaves the
softmax unchanged, and expand $e^{(z_j - \bar z)/\tau} \approx 1 + (z_j - \bar
z)/\tau$ to first order. The denominator becomes $k$ because the deviations
sum to zero, so $\mathrm{softmax}(\mathbf{z}/\tau)_j \approx \tfrac{1}{k}\big(1
+ (z_j - \bar z)/\tau\big)$. The difference $q_j^{(\tau)} - p_j^{(\tau)}$
therefore shrinks like $1/\tau$, and the gradient like $1/\tau^2$:

$$
\frac{1}{\tau}\big(q_j^{(\tau)} - p_j^{(\tau)}\big)
\;\approx\; \frac{1}{k\,\tau^2}\Big(\big(z^{\textrm{stu}}_j - \bar z^{\textrm{stu}}\big)
- \big(z^{\textrm{tea}}_j - \bar z^{\textrm{tea}}\big)\Big).
$$

Multiplying the loss by $\tau^2$ therefore keeps the size of the gradient
roughly constant as the temperature is tuned. This matters because the
distillation loss is rarely used alone. In practice it is added, with a fixed
weight, to the ordinary cross-entropy on the recorded labels, and without the
factor every change of $\tau$ would also change the balance between the two
terms. With the factor, the temperature decides only how much of the teacher's
small probabilities the student gets to see.

We check the scaling on teacher logits $(5, 2, -1)$ and student logits $(3, 3,
0)$. Both have mean $2$, so $\tau^2$ times the gradient should approach
$\tfrac13\big((1, 1, -2) - (3, 0, -3)\big) = (-0.667, 0.333, 0.333)$, a vector
of norm $0.816$. We let automatic differentiation compute the gradient of the
KL term at five temperatures, and print its norm without and with the factor
$\tau^2$.

In [7]:
z_tea = tf.constant([5.0, 2.0, -1.0])
z_stu = tf.Variable([3.0, 3.0, 0.0])
print(' tau  |grad|  |tau^2 grad|  tau^2 grad')
for tau in (1.0, 2.0, 5.0, 10.0, 20.0):
    with tf.GradientTape() as tape:
        p_tau = tf.nn.softmax(z_tea / tau)
        kl = tf.reduce_sum(
            p_tau * (tf.math.log(p_tau) - tf.nn.log_softmax(z_stu / tau)))
    g = tape.gradient(kl, z_stu).numpy()
    print(f'{tau:4.0f}  {np.linalg.norm(g):.4f}  '
          f'{np.linalg.norm(tau**2 * g):12.3f}  {np.round(tau**2 * g, 3)}')
limit = (z_stu - tf.reduce_mean(z_stu)) - (z_tea - tf.reduce_mean(z_tea))
limit = limit.numpy() / len(limit)
print(f'limit {"":6s}  {np.linalg.norm(limit):12.3f}  {np.round(limit, 3)}')

 tau  |grad|  |tau^2 grad|  tau^2 grad
   1  0.6391         0.639  [-0.462  0.441  0.022]
   2  0.2190         0.876  [-0.672  0.549  0.123]
   5  0.0368         0.920  [-0.741  0.478  0.263]
  10  0.0088         0.884  [-0.719  0.413  0.306]
  20  0.0021         0.854  [-0.697  0.374  0.322]
limit                0.816  [-0.667  0.333  0.333]


Without the factor the gradient collapses. Its norm falls from $0.64$ at $\tau
= 1$ to $0.0021$ at $\tau = 20$, about three hundred times smaller, and the
last doubling of $\tau$ divides it by about four, as $1/\tau^2$ predicts. With
the factor, the norm stays between $0.64$ and $0.92$. It overshoots the
predicted limit $0.816$, peaks near $\tau = 5$, and comes back slowly, at a
rate of order $1/\tau$: at $\tau = 20$ it still reads $0.854$. At $\tau = 1$
the loss is the ordinary KL divergence, and with a one-hot teacher, which has
zero entropy, it is the ordinary cross-entropy loss, so standard training is a
special case.

## Entropy and KL Divergence for Densities

So far every outcome could be counted. A regression target is a real number
with a density, and the sums become integrals. The KL divergence survives
this change. The entropy does not: it can be negative, and it comes to depend
on the units in which the variable is measured. The loss of a continuous
model can therefore fall below zero, and its floor moves with the units. What
keeps its meaning is the excess over the floor, and with it the difference
between two models on the same data. A first reading can skip these three
subsections and resume at the last paragraph of the section.

### Differential Entropy

For a continuous random variable $X$ with density $p$, the sum in the entropy
becomes an integral. The result is called the *differential entropy*,

$$
h(X) = -\int p(x) \log p(x) \,dx.
$$

It is the average surprisal of a density value, and density values are not
probabilities: they carry the units of $1/x$, and they can exceed one. We
write a lowercase $h$ to keep it apart from the discrete $H$, because the two
behave differently.

The smallest case shows how. The uniform density on $[0, a]$ equals $1/a$
there, so $h = -\int_0^a \tfrac1a \log \tfrac1a \,dx = \log a$. For
$a = \tfrac12$ this is $-\ln 2 \approx -0.693$ nats, a negative entropy, which
no discrete distribution can have. Measuring the same variable in centimetres
instead of metres multiplies $a$ by $100$ and adds $\ln 100 \approx 4.61$
nats, although the uncertainty has not changed.

A shift of this kind occurs under any smooth change of variables. If $y = g(x)$
is invertible with derivative $g'$, the density of $Y$ at $g(x)$ is
$p_X(x)/\lvert g'(x)\rvert$, and so

$$
h(Y) = h(X) + \mathbb{E}\big[\log\lvert g'(X)\rvert\big].
$$

For a vector $\mathbf{x} \in \mathbb{R}^n$ the definition and the shift
keep their form, with an integral over $\mathbb{R}^n$ and the Jacobian
determinant $\lvert\det \partial g/\partial \mathbf{x}\rvert$ in place of
$\lvert g'\rvert$. Stretching a variable raises its differential entropy,
and squeezing it lowers the entropy without limit. A Gaussian with standard
deviation $\sigma$ has $h = \tfrac12 \log(2\pi e \sigma^2)$, as we compute in
that section, which is negative once $\sigma <
(2\pi e)^{-1/2} \approx 0.24$.

The KL divergence does not share this defect. Apply the same change of
variables to a model density $q$ as to the data: both densities are divided by
the same factor, which cancels in the ratio $p/q$, so the KL divergence does
not change. For two uniform densities $p_1$ on $[0, a_1]$ and $p_2$ on $[0,
a_2]$ with $a_1 \le a_2$, for instance, it is $\int p_1 \log(p_1/p_2)\,dx =
\log(a_2/a_1)$, which for $a_1 = \tfrac12$ and $a_2 = 1$ is $\ln 2 \approx
0.693$ nats in metres and in centimetres alike. The cross-entropy
$-\mathbb{E}_P[\log q]$, on which a continuous model is trained, shifts by the
same constant as $h$. The constant does not depend on the model, so the
gradients and the minimizer of the cross-entropy are unchanged. When $h(p)$ is
finite, the decomposition the equation holds with $h$ in place of
$H$, and maximum likelihood for densities minimizes
$D_{\textrm{KL}}(p\|q_{\boldsymbol\theta})$ at the population level.

Differential entropy is not the limit of discrete entropy. Divide the line
into cells of width $\Delta$, and let $X_\Delta$ record which cell contains
$X$. Then $P(X_\Delta = i) \approx p(x_i) \Delta$ for a point $x_i$ in cell
$i$, and

$$
H(X_\Delta) + \log \Delta \to h(X) \quad \textrm{as } \Delta \to 0
$$

for densities regular enough that the sums converge to the integrals (for
example, when $p$ is continuous and $p$ and $p \log p$ are Riemann-integrable).
The discrete entropy itself grows like $\log(1/\Delta)$, since a finer grid
takes more questions, and $h$ is what remains after that growth is subtracted.
For the uniform density on $[0, 1]$ with $\Delta = \tfrac18$, the eight cells
are equally likely, $H_2(X_\Delta) = 3$ bits and $\log_2 \Delta = -3$, which
sum to $0 = h$. Exercise 9
proves the limit and checks it on a Gaussian.

### KL Divergence Between Gaussians

When a model reports a real number together with its uncertainty, it usually
reports a Gaussian: a mean and a variance. To compare such a model with the
data, or two such models with each other, we need the KL divergence between
two Gaussians. It has a closed form. For
$P = \mathcal{N}(\mu_1, \sigma_1^2)$ and $Q = \mathcal{N}(\mu_2, \sigma_2^2)$,

$$
D_{\textrm{KL}}\!\left(\mathcal{N}(\mu_1,\sigma_1^2) \,\big\|\, \mathcal{N}(\mu_2,\sigma_2^2)\right)
= \log\frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac{1}{2}.
$$

The divergence is a log ratio of standard deviations plus the variance and the
squared mean gap of $P$, both measured in units of the model's variance. It is
non-negative and zero only when $\mu_1 = \mu_2$ and $\sigma_1 = \sigma_2$, as
Gibbs' inequality requires. It is not symmetric, because $\sigma_1$ and
$\sigma_2$ enter differently. Exercise
10 derives the formula from the
definition.

The smallest case is a unit mean shift with equal variances, $P =
\mathcal{N}(0, 1)$ and $Q = \mathcal{N}(1, 1)$, where the formula gives
$(\mu_1 - \mu_2)^2/(2\sigma^2) = \tfrac12$ nat in either direction. Now let
the variances differ: $P = \mathcal{N}(0,1)$ and $Q = \mathcal{N}(0,4)$, so
$\sigma_2 = 2$. The formula gives $D_{\textrm{KL}}(P\|Q) = \ln 2 + \tfrac18 -
\tfrac12 \approx 0.318$ and $D_{\textrm{KL}}(Q\|P) = -\ln 2 + 2 - \tfrac12
\approx 0.807$ nats, a factor of $2.5$ apart.

the figure shows where each of the two divergences comes
from. In both directions the integrand changes sign at $|x| = (8 \ln 2/3)^{1/2}
\approx 1.36$, where the two densities cross. In panel (a) the wide model
explains narrow data. It is too surprised in the core $|x| < 1.36$, which
contributes $+0.424$, and less surprised than the truth in the tails, which
contribute $-0.106$. In panel (b) the narrow model has to explain wide data.
The core contributes $-0.240$, and the tails, where the narrow model gives
little probability to outcomes that occur, contribute $+1.046$.

![Where the KL divergence between $\mathcal{N}(0,1)$ and $\mathcal{N}(0,4)$ comes from. (a) The narrow distribution explained by the wide model: a positive core and slightly negative tails sum to $0.318$ nats. (b) The wide distribution explained by the narrow model: the tails, where the narrow model assigns little probability, contribute $1.05$ nats against a core of $-0.24$, for a total of $0.807$. The sign changes at $|x| = 1.36$ in both directions.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-it-kl-integrand.svg)

A narrow model is far more surprised by wide data than a wide model is by
narrow data. The KL divergence grows large when outcomes occur that the model
has nearly ruled out, and it stays small when the model merely spreads
probability over outcomes that do not occur. What this asymmetry does to a
model that is fitted in one direction or the other is the subject of
that section.

The definition suggests a second way to compute a KL divergence: draw samples
from $P$ and average $\log p(x) - \log q(x)$. This is how a divergence is
estimated when no closed form exists, so it is worth knowing how accurate
such averages are. The next cell defines the closed form and the Monte Carlo
estimate with its standard error.

In [8]:
def gaussian_kl(mu1, sigma1, mu2, sigma2):
    """Closed-form KL(N(mu1, sigma1^2) || N(mu2, sigma2^2)), in nats."""
    return (math.log(sigma2 / sigma1)
            + (sigma1 ** 2 + (mu1 - mu2) ** 2) / (2 * sigma2 ** 2)
            - 0.5)

def gaussian_log_pdf(x, mu, sigma):
    """Log-density of N(mu, sigma^2) at x."""
    return (-0.5 * math.log(2 * math.pi * sigma ** 2)
            - (x - mu) ** 2 / (2 * sigma ** 2))

def mc_kl(mu1, sigma1, mu2, sigma2, n=200000):
    """Monte Carlo estimate of the same KL from n samples x ~ P, with its
    standard error."""
    x = np.random.default_rng().normal(loc=mu1, scale=sigma1, size=n)
    r = gaussian_log_pdf(x, mu1, sigma1) - gaussian_log_pdf(x, mu2, sigma2)
    return float(r.mean()), float(r.std() / math.sqrt(n))

For the mean shift the accuracy can be predicted. The log-ratio is $\log p(x)
- \log q(x) = \tfrac12 - x$, which has variance one under $P$, so the average
of $n = 2 \cdot 10^5$ samples has standard error $1/\sqrt{n} = 0.0022$.

For the pair with different variances, the two directions are not equally
easy to estimate. Both average the same function up to its sign, $\ln 2 -
\tfrac38 x^2$, one over samples from the narrow Gaussian and the other over
samples from the wide one, where $x^2$ fluctuates four times as much. We
estimate all three divergences.

In [9]:
pairs = {'N(0,1) || N(1,1)': (0.0, 1.0, 1.0, 1.0),
         'N(0,1) || N(0,4)': (0.0, 1.0, 0.0, 2.0),
         'N(0,4) || N(0,1)': (0.0, 2.0, 0.0, 1.0)}
print(f'{"P || Q":18s} {"closed":>7s} {"sampled":>8s} {"std err":>8s}')
for name, args in pairs.items():
    estimate, std_err = mc_kl(*args)
    print(f'{name:18s} {gaussian_kl(*args):7.4f} {estimate:8.4f} {std_err:8.4f}')

P || Q              closed  sampled  std err


N(0,1) || N(1,1)    0.5000   0.4990   0.0022
N(0,1) || N(0,4)    0.3181   0.3184   0.0012
N(0,4) || N(0,1)    0.8069   0.8049   0.0047


The standard errors are $0.0022$, $0.0012$ and $0.0047$, and every estimate
lies within a few of its standard errors of the closed form. The direction that
draws from the wide Gaussian is four times noisier than its reverse, so it
needs sixteen times as many samples for the same accuracy. Panel (b) of
the figure shows why: this divergence is collected far out
in the tails, where the log-ratio is large and the samples thin out.
that section meets the same difficulty at a much larger
scale.

### Maximum Entropy and the Gaussian

On a finite set the uniform distribution is the most uncertain one. Which
density on the real line is? Without a constraint there is none. No density is
uniform on $\mathbb{R}$, and stretching a density raises its differential
entropy without limit. So we fix the two numbers that a Gaussian model reports,
the mean $\mu$ and the variance $\sigma^2$. Among all densities with these two
moments, the Gaussian has the largest entropy. This is the sense in which a
Gaussian noise model assumes the least: it commits to a mean and a variance,
and to nothing else.

The proof is the one-line argument for the uniform bound, with one change.
There $-\log u(x) = \log k$ was the same for every outcome, so every
distribution $P$ had the same cross-entropy $\log k$ to the uniform model.
Here let $\varphi$ be the Gaussian density with the given moments. Its
surprisal $-\log \varphi(x) = \tfrac{1}{2}\log(2\pi\sigma^2) +
\tfrac{(x-\mu)^2}{2\sigma^2}$ is not constant, but it is a quadratic in $x$,
and the average of a quadratic depends on $P$ only through the two moments
that we fixed. For every density $P$ with these moments,

$$
\textrm{CE}(P, \varphi)
= \tfrac{1}{2}\log(2\pi\sigma^2) + \frac{\mathbb{E}_P\big[(x-\mu)^2\big]}{2\sigma^2}
= \tfrac{1}{2}\log(2\pi\sigma^2) + \tfrac{1}{2}
= \tfrac{1}{2}\log(2\pi e \sigma^2).
$$

The cross-entropy to the Gaussian does not depend on which such $P$ produced
the data. For $P = \varphi$ the cross-entropy is the entropy, so this value is
the Gaussian's own differential entropy, the closed form quoted whenever a
Gaussian entropy is needed. For every other such $P$, the decomposition
the equation, with $h$ in place of $H$, and Gibbs' inequality
give

$$
h(P) = \textrm{CE}(P, \varphi) - D_{\textrm{KL}}(P\|\varphi)
\leq \tfrac{1}{2}\log(2\pi e \sigma^2) = h(\mathcal{N}(\mu, \sigma^2)),
$$

with equality if and only if $P = \mathcal{N}(\mu, \sigma^2)$. (If $h(P) =
-\infty$, there is nothing to prove.) The gap in this inequality,
$D_{\textrm{KL}}(P\|\varphi)$, has a use of its own: it is the excess loss of
the best Gaussian model on data that are not Gaussian (Exercise
11).

The uniform and the Gaussian are two cases of one mechanism. Whenever the
surprisal of a candidate averages to the same value under every distribution
that meets the constraints, the candidate has the largest entropy among them.
that section proves this for constraints on the
averages of any set of statistics, where the candidates form an exponential
family. The geometric distribution on the counts $\{0, 1, 2, \ldots\}$ is a
third case. Its surprisal is affine in the outcome, so it has the largest
entropy among the distributions with its mean.

The model $Q'$ gave zero probability to an outcome that occurs, and its KL
divergence was infinite; the narrow Gaussian of the figure
came close, with a divergence that arose in tails it had nearly ruled out. A
model that turns a few random numbers into an image puts all its probability
on a thin surface in the space of pixels. If the data do not lie on that
surface, the KL divergence is infinite whether the surface passes close by
the data or far from them. that section shows that
every divergence built on the ratio $p/q$ shares this blindness, and
that section builds comparisons that still tell
near from far. A second question is closer at hand. We counted questions and
read them as codewords, but only for a source whose surprisals are whole
numbers of bits. that section builds a code for every
source and asks how close a real compressor gets to the entropy.

## Summary

A cross-entropy loss is the model's average surprisal at the data. It is the
entropy of the data, which no model can lower, plus the KL divergence from the
data to the model, which by Gibbs' inequality is never negative and vanishes
only for the right model. A loss must therefore be read against its floor, the
conditional entropy of the label given the input, which on fresh data only
the true conditional probabilities reach. The floor is zero when the input
determines the label, and $H_b(\rho) + \rho \log(k-1)$ under symmetric label
noise. In bits, the entropy of a finite source is the average number of yes/no
questions needed to identify an outcome, to within one question, and it is at
most $\log_2 k$ for $k$ outcomes. Softened targets move the optimum: label
smoothing to a finite logit gap with a floor above zero, distillation to the
softened teacher, where the factor $\tau^2$ holds the gradient scale fixed as
the temperature changes. For densities the entropy depends on the units of the
data, while the KL divergence and the minimizer of the cross-entropy do not.

## Exercises

1. **Uniqueness of self-information.** ● Show that self-information is
   unique up to scale: if a strictly decreasing function $I(p)$ on $(0, 1]$
   satisfies $I(1) = 0$ and $I(p_1 p_2) = I(p_1) + I(p_2)$ for all
   $p_1, p_2 \in (0, 1]$, then $I(p) = -c \log p$ for some constant
   $c > 0$. Work with the function $g(t) = I(e^{-t})$ for $t \geq 0$.

1. **Two proofs of Gibbs' inequality.** The text proves Gibbs' inequality
   with Jensen's inequality. that section
   proves it from the tangent line $\ln u \le u - 1$.
    1. Show that the tangent-line proof, with $u = q(x)/p(x)$ on the support
       of $P$, gives $D_{\textrm{KL}}(P\|Q) \ge 1 - Q(\textrm{supp}\,P)$,
       where the text obtains $-\log Q(\textrm{supp}\,P)$. Which bound is
       larger?
    1. Find a pair $P \neq Q$ for which the bound of the text holds with
       equality. Can the tangent-line bound hold with equality for
       $P \neq Q$?
    1. Why are the two proofs nevertheless the same argument? Hint: the
       standard proof of Jensen's inequality for a concave $f$ bounds $f$ by
       its tangent line at $\mathbb{E}[X]$.

1. [code] **The label-noise floor.** Labels on $k$ classes are correct with
   probability $1-\rho$ and otherwise uniform over the other $k-1$ classes,
   independently of the input.
    1. Assume that the clean label is a function of the input. Show that the
       population cross-entropy of any classifier is at least
       $H_b(\rho) + \rho\log(k-1)$, that the optimal prediction is the noisy
       conditional distribution itself, and that the optimal accuracy is
       $1-\rho$ whenever $1 - \rho > \rho/(k-1)$.
    1. Evaluate the floor for $k = 10$ and $\rho = 0.05, 0.1, 0.2$.
    1. A colleague reports a validation loss of $0.40$ nats on a 10-class
       data set known to have $10\%$ such noise in every split. List
       possible explanations, and say which of them the floor rules out and
       which it does not.
    1. Train the softmax regression of that section on
       Fashion-MNIST with $10\%$ of the training and test labels flipped in
       this way. Compare the test loss with $0.545$ and the test accuracy with
       $90\%$. The gap is large. Which assumption of (a) does Fashion-MNIST
       violate, what does the linear model's own error add, and how could
       you estimate the floor that does apply?

1. **Label noise on an ambiguous input.** The floor of the previous exercise
   assumes that the input determines the clean class. Fix one input at which
   it does not: the clean class $C$ has the distribution $\mathbf{c}$ on $k$
   classes, and the recorded label $Y$ equals $C$ with probability $1 - \rho$
   and is otherwise uniform over the other $k - 1$ classes.
    1. Show that $H(Y \mid C) = H_b(\rho) + \rho\log(k-1)$, whatever
       $\mathbf{c}$ is. Call this value $H_{\textrm{noise}}$.
    1. Write the chain rule the equation for the pair
       $(C, Y)$ in both orders, and conclude that
       $H_{\textrm{noise}} \le H(Y) \le H(C) + H_{\textrm{noise}}$. Which
       result of the text gives the lower bound? For which $\mathbf{c}$ and
       $\rho$ does the upper bound hold with equality?
    1. For $k = 2$ and $\rho = 0.1$, compute $H(Y)$ for
       $\mathbf{c} = (1, 0)$, $(0.9, 0.1)$ and $(\tfrac12, \tfrac12)$, and
       compare each value with the two bounds. How much does the noise add
       to the entropy of the label when the clean class is a fair coin, and
       why?
    1. A data set has $10\%$ label noise on ten classes, and some of its
       inputs are ambiguous. Is the value $0.545$ nats of
       that section a lower or an upper bound on
       its floor?

1. [code] **The initial-loss check.** A freshly initialized network produces
   small, random logits.
    1. Let the $k$ logits be independent with mean $0$ and variance $s^2$.
       Expand $\log \sum_j e^{z_j}$ to second order in the $z_j$ to show
       $\mathbb{E}[\textrm{CE}] \approx \ln k + \tfrac{s^2}{2}\big(1 -
       \tfrac1k\big)$.
    1. Check the formula by sampling Gaussian logits for $k = 10$ and
       $s = 0.1, 0.5, 1$. Where does the second-order approximation start to
       fail, and in which direction?
    1. A 10-class model reports a loss of $4.1$ at initialization. What does
       the formula say about the scale of its logits, and what would you
       change in the initialization? Given what you found in (b), is the
       formula still trustworthy at that scale?

1. **Other scoring rules.** A scoring rule $S(\mathbf{q}, y)$ is strictly
   proper if the expected penalty
   $\mathbb{E}_{y \sim \mathbf{p}}[S(\mathbf{q}, y)]$ is uniquely minimized
   at $\mathbf{q} = \mathbf{p}$. For the log score, this property is Gibbs'
   inequality.
    1. Show that the expected Brier score, with
       $S(\mathbf{q}, y) = \sum_j (q_j - \mathbf{1}(j = y))^2$, exceeds the
       expected Brier score of the truthful report by
       $\|\mathbf{q} - \mathbf{p}\|^2$, so that the Brier score is strictly
       proper [@Gneiting.Raftery.2007].
    1. The linear score $S(\mathbf{q}, y) = -q_y$ rewards the probability
       placed on the observed class. Which report minimizes its expected
       penalty? Is the linear score proper? What does a classifier trained
       with it report for an input whose label is class $1$ with probability
       $0.6$ and class $2$ with probability $0.4$?

1. **Label smoothing on ambiguous inputs.** ● Label smoothing trains with
   the target the equation.
    1. In the training distribution, an input $\mathbf{x}$ carries the
       label $y$ with probability $p(y \mid \mathbf{x})$. Show that the
       expected smoothed loss for this input is minimized by
       $\mathbf{q}^*(\cdot \mid \mathbf{x}) = (1 - \epsilon)\,
       p(\cdot \mid \mathbf{x}) + \tfrac{\epsilon}{k}\mathbf{1}$. How can
       $p(\cdot \mid \mathbf{x})$ be recovered from $\mathbf{q}^*$? What is
       the largest probability that the minimizer can assign to a class?
    1. For one example with logits $\mathbf{z}$ and
       $\mathbf{q} = \mathrm{softmax}(\mathbf{z})$, the gradient of
       $\textrm{CE}(\mathbf{p}^\epsilon, \mathbf{q})$ with respect to $z_j$
       is $q_j - p^\epsilon_j$. For $\epsilon > 0$, show that all
       wrong-class logits are equal at every stationary point.
    1. With one-hot targets ($\epsilon = 0$), the gradient with respect to a
       wrong-class logit is $q_j$, and the loss has no stationary point.
       Show that gradient descent on the logits of one example still shrinks
       the gap between two wrong-class logits, and explain why this
       shrinkage stalls as $q_y \to 1$. In a network, the logits of all
       examples are computed from shared features. Which differences between
       wrong-class logits can therefore survive training with
       $\epsilon = 0$? Why does training with $\epsilon > 0$ remove them?
    1. @Muller.Kornblith.Hinton.2019 observe that teachers trained
       with label smoothing produce worse students under the distillation
       loss the equation. Explain this observation with the
       results of (b) and (c).

1. [code] **Distillation at high temperature.** ●
   @Hinton.Vinyals.Dean.2015 state in their Section 2.1 that, at
   high temperature, distillation reduces to matching zero-mean logits by
   squared error.
    1. Write $q_j = p_j(1 + u_j)$, so that $\sum_j p_j u_j = 0$, and expand
       $D_{\textrm{KL}}(\mathbf{p} \| \mathbf{q}) = -\sum_j p_j \log(1 + u_j)$
       to second order to obtain

       $$
       D_{\textrm{KL}}(\mathbf{p} \| \mathbf{q}) =
       \frac{1}{2}\sum_j \frac{(q_j - p_j)^2}{p_j}
       + O\big(\|\mathbf{q} - \mathbf{p}\|^3\big).
       $$

       Combine this expansion with the large-$\tau$ approximation of
       $\mathrm{softmax}(\mathbf{z}/\tau)$ in
       that section to show that, as $\tau \to \infty$,

       $$
       \tau^2 D_{\textrm{KL}}\big(\mathbf{p}^{(\tau)} \,\|\, \mathbf{q}^{(\tau)}\big)
       \to \frac{1}{2k} \sum_{j=1}^k
       \Big(\big(z^{\textrm{stu}}_j - \bar z^{\textrm{stu}}\big)
       - \big(z^{\textrm{tea}}_j - \bar z^{\textrm{tea}}\big)\Big)^2.
       $$

    1. Evaluate $\tau^2 D_{\textrm{KL}}(\mathbf{p}^{(\tau)} \| \mathbf{q}^{(\tau)})$
       for $\tau = 1, 2, 5, 10, 100$ with the logits `z_tea` $= (5, 2, -1)$
       and `z_stu` $= (3, 3, 0)$ of that section, and
       compare it with the limit.
    1. At low temperature, which teacher logits have almost no influence on
       the loss? @Hinton.Vinyals.Dean.2015 argue that this can be an
       advantage and also a drawback. Give one argument for each side,
       based on what the teacher's own cross-entropy training does to these
       logits.

1. [code] **Quantized entropy against differential entropy.** Let $X$ have
   density $p$, and let $X_\Delta$ record which cell
   $[i\Delta, (i+1)\Delta)$ contains $X$.
    1. Show that $H(X_\Delta) + \log \Delta \to h(X)$ as $\Delta \to 0$ when
       $p$ is continuous and $p$ and $p \log p$ are Riemann-integrable. Write
       $P(X_\Delta = i) = p(x_i)\Delta$ for some $x_i$ in the cell, by the
       mean value theorem for integrals.
    1. Check the limit by hand for the uniform density on $[0, 1]$ with
       $\Delta = \tfrac18$. What happens for a cell width that does not
       divide $1$, such as $\Delta = 0.3$?
    1. Compute $H(X_\Delta) + \log \Delta$ numerically for
       $X \sim \mathcal{N}(0, 1)$ with $\Delta = 0.5, 0.1, 0.01$, using the
       Gaussian distribution function for the cell probabilities, and compare
       it with $h = \tfrac12\log(2\pi e) \approx 1.4189$ nats. (At
       $\Delta = 0.5$ you should find $1.4292$.)
    1. Why does $H(X_\Delta)$ itself diverge as $\Delta \to 0$? A regression
       target is binned into cells of width $\Delta$ and predicted with a
       softmax classifier. Which quantity does its loss approach at best,
       and what must be added to it before it can be compared with the
       negative log-density of a continuous model?

1. **Gaussian KL divergence.** Let $P = \mathcal{N}(\mu_1, \sigma_1^2)$.
    1. Derive the closed form the equation by writing out
       $\mathbb{E}_{x\sim P}[\log p(x) - \log q(x)]$ and using
       $\mathbb{E}_{x \sim P}[x] = \mu_1$ and
       $\mathbb{E}_{x\sim P}[(x-\mu_1)^2] = \sigma_1^2$.
    1. Let $Q$ have mean $\mu_1 + \delta$ and standard deviation
       $\sigma_1 e^{\eta}$. Expand both $D_{\textrm{KL}}(P\|Q)$ and
       $D_{\textrm{KL}}(Q\|P)$ to second order in $\delta$ and $\eta$. At
       which order do the two directions first differ?
    1. Evaluate the second-order approximation for $P = \mathcal{N}(0, 1)$
       and $Q = \mathcal{N}(0, 4)$, and compare it with the exact values
       $0.318$ and $0.807$ nats of the two directions.

1. **The excess loss of a Gaussian model.** A regression model reports a
   Gaussian $\mathcal{N}(m, s^2)$ for a target whose density $P$ has mean
   $\mu$, variance $\sigma^2$ and a finite differential entropy.
    1. Show that $\textrm{CE}(P, \mathcal{N}(m, s^2))$ is smallest at
       $m = \mu$ and $s^2 = \sigma^2$, and that this best Gaussian model
       exceeds the floor $h(P)$ by
       $D_{\textrm{KL}}(P\|\varphi) = h(\varphi) - h(P)$, where $\varphi$ is
       the density of $\mathcal{N}(\mu, \sigma^2)$.
    1. Evaluate the excess for the uniform density on $[0, a]$, which has
       variance $a^2/12$, and for the Laplace density
       $\tfrac{1}{2b} e^{-|x|/b}$, which has variance $2b^2$ and
       differential entropy $1 + \log(2b)$. (You should find about $0.176$
       and $0.072$ nats.)
    1. Neither answer depends on the scale $a$ or $b$. Explain why without
       computing, from what that section says
       about a change of units.
    1. The Gaussian has the largest entropy for a given variance, so no
       density has a higher floor. Is there also a largest excess? Consider
       densities that concentrate near the two points $\mu \pm \sigma$.
       What does your answer say about a Gaussian model for a target with
       two well-separated modes?

[Discussions](https://d2l.discourse.group/t/1104)